In [1]:
import math
from datetime import timedelta
from pathlib import Path

import numpy as np
import polars as pl
import yaml

with open("../config.budapest.yaml") as fp:
    config = yaml.safe_load(fp)

In [2]:
def entropy(data: pl.DataFrame, result_column: str = "entropy") -> pl.DataFrame:
    result = np.nan

    d = data.clone()
    place_counts = d.group_by("place_label").len()
    N = place_counts["len"].sum()
    if len(data) > 1:
        place_counts = place_counts.with_columns(
            (pl.col.len / place_counts["len"].sum()).alias("probability")
        ).with_columns((pl.col.probability * pl.col.probability.log10()).alias("part"))

        result = -place_counts["part"].sum() / math.log10(N)

    return pl.DataFrame(
        {
            "device_id": [data["device_id"][0]],
            result_column: [result],
            "number_of_places": N,
        }
    )

In [3]:
DIR = f"{config['dirs']['output']}/{config['city']}"

In [4]:
test = pl.DataFrame(
    {
        "device_id": [1, 1, 1, 1, 1],
        "place_label": [1, 1, 2, 2, 2],
        "interval_lenght": [
            timedelta(minutes=20),
            timedelta(minutes=30),
            timedelta(minutes=25),
            timedelta(minutes=15),
            timedelta(minutes=10),
        ],
        "latitude_mean": [
            47.6612,
            47.6612,
            47.4984,
            47.4984,
            47.4984,
        ],
        "longitude_mean": [19.0764, 19.0764, 19.0404, 19.0404, 19.0404],
    }
)

In [29]:
records = [
    [42, 1, 4],
    [42, 1, 4],
    [42, 1, 4],
    [42, 1, 4],
    [42, 1, 4],
    [42, 2, 4],
    [42, 3, 4],
    [42, 3, 4],
    [42, 3, 4],
    [42, 5, 4],
    [42, 5, 4],
    [42, 5, 4],
    [42, 5, 4],
    [42, 5, 4],
    [42, 5, 4],
    [42, 5, 4],
    [42, 4, 4],
    [51, 10, 4],
    [51, 10, 4],
    [52, 11, 4],
    [52, 12, 4],
    [52, 13, 4],
    [52, 14, 4],
    [53, 10, 4],
    [53, 10, 4],
    [53, 10, 4],
    [53, 11, 4],
]
test2 = pl.from_records(
    records, schema=["device_id", "place_label", "home"], orient="row"
)

In [33]:
test.group_by("device_id").map_groups(entropy)


device_id,entropy
i64,f64
1,0.418166


In [46]:
test2.group_by("device_id").map_groups(entropy)

device_id,entropy,number_of_places
i64,f64,i32
53,0.405639,4
52,1.0,4
42,0.481686,17
51,-0.0,2


In [47]:
actual = {}
for row in test2.group_by("device_id").map_groups(entropy).iter_rows(named=True):
    actual[row["device_id"]] = row["entropy"]

In [48]:
expected = {42: 0.48, 51: 0, 52: 1, 53: 0.4}

In [49]:
for k in actual:
    np.testing.assert_array_almost_equal(actual[k], expected[k], 2)

In [ ]:
Path(f"{DIR}/descriptives/entropy/").mkdir(exist_ok=True, parents=True)
for i in range(128, 256):
    temp = pl.read_parquet(f"{DIR}/stops/v2/{i:02x}.parquet")
    ent = temp.group_by("device_id").map_groups(entropy)
    ent.sort("device_id").write_csv(f"{DIR}/descriptives/entropy/{i:02x}.csv")
# 90s  for 16 partitions
# 10m 45s for 112 partitions
# 13m 30s for 128 partitions

In [55]:
acc = pl.DataFrame()
for i in range(256):
    g = pl.read_csv(f"{DIR}/descriptives/entropy/{i:02x}.csv")
    acc = pl.concat([acc, g])
acc = acc.sort("device_id")
acc.write_csv(f"{DIR}/descriptives/entropy/full.csv")

In [ ]:
acc = pl.DataFrame()
for i in range(256):
    temp = (
        pl.read_parquet(f"{DIR}/stops/v2/{i:02x}.parquet")
        .with_columns(pl.col("interval_start").dt.weekday().alias("dow"))
        .filter(pl.col.dow > 5)
    )
    ent = temp.group_by("device_id").map_groups(entropy)
    acc = pl.concat([acc, ent])
acc.write_csv(f"{DIR}/descriptives/entropy/full_weekends.csv")
# 18m

In [ ]:
acc = pl.DataFrame()
for i in range(256):
    temp = (
        pl.read_parquet(f"{DIR}/stops/v2/{i:02x}.parquet")
        .with_columns(pl.col("interval_start").dt.weekday().alias("dow"))
        .filter(pl.col.dow <= 5)
    )
    ent = temp.group_by("device_id").map_groups(entropy)
    acc = pl.concat([acc, ent])
acc.write_csv(f"{DIR}/descriptives/entropy/full_weekdays.csv")
# 26m